# Pelot et al. (2020), vagus nerve morphology of human, pig and rat, on SPARC:60

**Paper.** Pelot, Goldhagen, Cariello, Musselman, Clissold, Ezzell and Grill (2020),
"Quantified Morphology of the Cervical and Subdiaphragmatic Vagus Nerves of Human, Pig,
and Rat", Front. Neurosci. 14, 601479, doi:10.3389/fnins.2020.601479. The claims file
(`claims/pelot-2020.yaml`) was written from the version of record (Europe PMC, PMC7672126),
without the supplement and with Eqs. 1 to 4 lost in the text.

**Dataset.** SPARC:60, "Quantified morphology of the rat vagus nerve", version 5
(doi:10.26275/lg3c-1kjf, published 2025-02-24), read from the public SPARC bucket at the
object versions its manifest gives. Version 5 is the latest and the version the run was
given; the paper cites none. Versions 1 to 4 date from 2020; the data check below compares
the files the analysis reads in version 5 with version 4 (2020-09-30), the version current
when the paper was published.

**Sister datasets.** The paper's human and pig data are in three other SPARC datasets that
list the paper as `IsDescribedBy`: SPARC:65 (human anti-claudin-1 IHC), SPARC:64 (pig
Masson's trichrome) and SPARC:89 (pig anti-fibronectin IF), the paper's Pelot et al. (2020a,
f and b). They are used for the claims the claims file marks `with_inputs` or `elsewhere`.
The paper's supplement cites SPARC:65 version 6 (doi:10.26275/nluu-1ews) and SPARC:89
version 1 (doi:10.26275/8pc2-rhu2), which are pinned; its reference list stops before
2020f, so SPARC:64 is pinned at its latest version, 4 (doi:10.26275/maq2-eii4, 2020-10-01). The
human values are recomputed here from the human traces; the pig values come from the
per-fascicle output files the paper's code wrote (`PigVNMorphology_OutputMetrics.mat`,
`PigIFMorphology.mat`), as recomputing the pig traces was out of the time available.

**Supplement.** The literature values of Supplement 4 (`Data_Sheet_2.xlsx`), which the
claims file's reader did not have, are read from the supplementary files of the version of
record in Europe PMC (PMC7672126) and checked against their SHA-256.

**Checksums.** Every file is checked against the SHA-256 of its version's manifest as it is
read; the last cell prints the SHA-256 of each file read.

**No random draws.** No analysis here draws random numbers, so no seed is needed and
nothing is seed sensitive.

| Claim | Tested here |
|---|---|
| rat-sample-set (link) | tested |
| nerve-diameter-anova | tested, with SPARC:65 and SPARC:64 |
| human-pig-nerve-diameter-similar | tested, on SPARC:65 and SPARC:64 |
| rat-nerve-diameter | tested |
| rat-ten-times-smaller | tested, with SPARC:65 and SPARC:64 |
| nerve-cervical-larger | tested, with SPARC:65 and SPARC:64 |
| fascicle-number-human-pig | tested, on SPARC:65 and SPARC:64 |
| rat-subdiaphragmatic-multifascicular | tested |
| rat-multifascicular-proportions | tested |
| fascicle-diameter-anova | tested, with SPARC:65 and SPARC:64 |
| rat-fascicle-diameter | tested |
| cervical-fascicle-diameter-species | tested, with SPARC:65 and SPARC:64 |
| subdiaphragmatic-fascicle-diameter-comparable | tested, with SPARC:65 and SPARC:64 |
| fascicles-larger-cervical | tested, with SPARC:65 and SPARC:64 |
| rat-diameter-like-fascicle | tested, with SPARC:65 and SPARC:64 |
| human-pig-tissue-proportions | tested, on SPARC:65, SPARC:64 and SPARC:89 |
| rat-level-ratios | tested |
| human-pig-level-ratios | tested, on SPARC:65 and SPARC:64 |
| human-variability | tested, with SPARC:65 and SPARC:64 |
| literature-comparison-human-pig | tested, with Supplement 4 and SPARC:65 and SPARC:64 |
| rat-literature-comparison | tested, with Supplement 4 |
| perineurium-species-order | tested, with SPARC:65 and SPARC:89 |
| perineurium-ratio-decreases | tested, with SPARC:65 and SPARC:89 |
| human-perineurium-fit | tested, on SPARC:65 |
| pig-perineurium-fit | tested, on SPARC:89 |
| rat-perineurium-fit | tested |
| perineurium-thicker-larger-fascicles | tested, on SPARC:65 and SPARC:89 |
| perineurium-no-level-difference | tested, with SPARC:65 and SPARC:89 |
| sunderland-reanalysis | tested, on the Sunderland and Bradley table in SPARC:65 |
| rat-perineurium-near-3-percent | tested |
| human-pig-perineurium-thicker | tested, on SPARC:65 and SPARC:89 |
| pig-if-mt-consistent | tested, on SPARC:89 and SPARC:64 |
| peanut-fascicle-counts | tested, on SPARC:65 and SPARC:89 |
| no-age-trend | tested, with SPARC:65 and SPARC:64 |
| rat-carotid-sheath-bundles | blocked (only the vagus is traced) |

In [ ]:
import io
import time
import json
import hashlib
import resource

import numpy as np
import pandas as pd
import requests
from scipy import stats

import vagus as v

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 70)
T0 = time.time()
print({ds: v.DATASETS[ds][:2] for ds in v.DATASETS})

## Data check

The dataset's files, its readme and the SDS sheets, every column.

In [ ]:
m60 = v.manifest("60")
print(m60["name"], "version", m60["version"], m60["datePublished"], m60["@id"])
print("related:", m60["relatedPublications"])
fl = pd.DataFrame(m60["files"])
fl["folder"] = fl["path"].str.split("/").str[:2].str.join("/")
print(fl.groupby("folder").agg(n=("path", "size"), bytes=("size", "sum")))
print(v.read("60", "readme.md").decode())
for p in ["files/derivative/README.txt", "files/code/README.txt"]:
    print("=====", p)
    print(v.read("60", p).decode())

In [ ]:
subjects = v.excel("60", "files/subjects.xlsx")
samples = v.excel("60", "files/samples.xlsx")
desc = v.excel("60", "files/dataset_description.xlsx", header=None)
print(desc.iloc[:, [0, 3, 4]].to_string())
print(subjects.to_string())
print(samples.drop(columns=["species", "strain", "RRID for strain"]).to_string())
for f in ["primary", "derivative", "code", "docs", "protocol"]:
    mf = v.excel("60", f"files/{f}/manifest.xlsx")
    print("=====", f, mf.shape, list(mf.columns))
    print(mf.iloc[:, :3].to_string(max_colwidth=90))

The rat samples. The level comes from `specimen anatomical location` in samples.xlsx.
Each sample is measured from its primary traces with the paper's method (`vagus.py`).

In [ ]:
smp = samples.merge(subjects[["subject_id", "age", "body mass"]], on="subject_id", suffixes=("", "_subj"))
smp["sub"] = smp["subject_id"].str[4:].astype(int)
smp["sam"] = smp["sample_id"].str[4:].astype(int)
smp["label"] = "R" + smp["sub"].astype(str) + "-" + smp["sam"].astype(str)
smp["level"] = np.where(smp["specimen anatomical location"].str.contains("cervical"), "cervical", "subdiaphragmatic")
smp["side"] = np.where(smp["specimen anatomical location"].str.startswith("left"), "left",
                       np.where(smp["specimen anatomical location"].str.contains("subdiaphragmatic"), "anterior (by readme)", "?"))
smp["age_days"] = smp["age_subj"].str.replace(" days", "").astype(int)
rat = {}
for _, row in smp.iterrows():
    rat[row["label"]] = v.rat_sample(row["sub"], row["sam"])
smp["n_fasc"] = [rat[l]["n_fasc"] for l in smp["label"]]
smp["um_per_px"] = [rat[l]["um_per_px"] for l in smp["label"]]
smp["image_px"] = [rat[l]["shape"] for l in smp["label"]]
smp["nerve_mask_used"] = [rat[l]["nerve_from_mask"] for l in smp["label"]]
print(smp[["label", "level", "side", "sex", "age_days", "body mass", "Scale Bar Length", "um_per_px", "image_px", "n_fasc", "nerve_mask_used"]].to_string())
print(pd.crosstab(smp["level"], smp["sex"], margins=True))
print("subjects:", smp["sub"].nunique(), "| sexes over subjects:", subjects["sex"].value_counts().to_dict())
print("age range (days):", subjects["age"].str.replace(" days", "").astype(int).agg(["min", "max"]).to_dict(),
      "| body mass (g): mean %.1f SD %.1f (n = %d)" % (subjects["body mass"].mean(), subjects["body mass"].std(), len(subjects)))
print("um per pixel range:", smp["um_per_px"].min().round(4), "to", smp["um_per_px"].max().round(4))
nm = [p for p in v.files("60") if p.endswith("NerveMask.tif")]
print("NerveMask files:", [p.split("/")[-1] for p in nm])

Duplicates: no two traces are identical (the SHA-256 of each PeriMask's pixels), and the
files the analysis reads are the same S3 objects (same object version and size) in version 4
(2020-09-30) as in version 5.

In [ ]:
hashes = {}
for _, row in smp.iterrows():
    pre = v._prefix("60", row["sub"], row["sam"])
    hashes[row["label"]] = hashlib.sha256(np.packbits(v.tif("60", pre + "PeriMask.tif")).tobytes()).hexdigest()
print("distinct PeriMask pixel arrays:", len(set(hashes.values())), "of", len(hashes))
br = requests.get("https://api.pennsieve.io/discover/datasets/60/versions/4/files/browse", params={"limit": 100}, timeout=60).json()
mv4 = [f["s3Version"] for f in br["files"] if f["path"] == "manifest.json"][0]
import fsspec
with fsspec.open(f"{v.BUCKET}/60/manifest.json?versionId={mv4}", block_size=2**20) as fh:
    m4 = json.load(fh)
print("version 4 manifest fields:", sorted(m4["files"][0]))
f4 = {f["path"]: (f.get("s3VersionId"), f["size"]) for f in m4["files"]}
used = [p for (ds, p) in v.READ if ds == "60" and p.startswith("files/primary")]
same = [p for p in used if f4.get(p) == (v.files("60")[p]["s3VersionId"], v.files("60")[p]["size"])]
print("version 4:", m4["version"], m4["datePublished"], "| primary files read:", len(used), "| identical in version 4:", len(same))
only5 = sorted(set(v.files("60")) - set(f4))
only4 = sorted(set(f4) - set(v.files("60")))
print("paths only in v5:", len(only5), only5[:8], "\npaths only in v4:", len(only4), only4[:8])
for p in ["files/code/RatVNMorphology_OutputMetrics.mat", "files/derivative/RatVNMorphology_OutputMetrics.mat"]:
    print(p, "same object in v4:", f4.get(p) == (v.files("60")[p]["s3VersionId"], v.files("60")[p]["size"]))

The measurements against the output files the paper's code wrote. The code folder's
`RatVNMorphology_OutputMetrics.mat` (written 2020-09-05) holds the 18 rat samples as the
statistics used them (`DataPlotting_v4.m`). The derivative folder's file of the same name
(written 2020-03-03) is another, older output.

In [ ]:
out_code = v.mat("60", "files/code/RatVNMorphology_OutputMetrics.mat")
out_der = v.mat("60", "files/derivative/RatVNMorphology_OutputMetrics.mat")
cmp = []
for i, lab in enumerate(out_code["sub_sam"]):
    pv = v.rat_plot_values(rat[lab])
    cmp.append(dict(sample=lab, d_nerve_here=pv["d_nerve"], d_nerve_code=out_code["dNerve_um"][i],
                    d_fasc_here=pv["d_fasc"], d_fasc_code=np.atleast_1d(out_code["dFasc_um"][i])[0]))
cmp = pd.DataFrame(cmp)
print(cmp.round(4).to_string())
print("max |difference|, nerve %.2e um, fascicle %.2e um" % ((cmp.d_nerve_here - cmp.d_nerve_code).abs().max(),
                                                           (cmp.d_fasc_here - cmp.d_fasc_code).abs().max()))
thk_code = np.concatenate([np.atleast_1d(t) for t in out_code["thkperi_um"]])
print("code file thkperi_um holds", len(thk_code), "values:", np.round(thk_code, 3))
print("derivative file: %d samples, prefixes %s" % (len(out_der["sub_sam"]), sorted(set(s[0] for s in out_der["sub_sam"]))))
der_r = {s: i for i, s in enumerate(out_der["sub_sam"]) if s.startswith("R")}
dd = pd.DataFrame([dict(sample=s, d_nerve_derivative=out_der["dNerve_um"][i], d_nerve_here_code_rule=v.rat_plot_values(rat[s])["d_nerve"],
                        d_nerve_here_trace=rat[s]["d_nerve"], n_fasc_derivative=out_der["NFasc"][i], n_fasc_here=rat[s]["n_fasc"])
                   for s, i in der_r.items()])
print(dd.round(2).to_string())

### The human and pig data

Human: the 18 samples of the paper are those of `CadaverVNMorphology_OutputMetrics.mat` in
SPARC:65's code folder (written 2020-09-05), each measured here from its traces
(PeriInnerMask, PeriExtMask, NerveMask), and checked against that file. Pig MT: the 18
samples of `PigVNMorphology_OutputMetrics.mat` in SPARC:64's code folder (nerve diameter,
every fascicle diameter, fascicle count). Pig IF: `PigIFMorphology.mat` in SPARC:89's code
folder (per sample, non-peanut fascicle diameters and perineurium thicknesses, and the
tissue proportions), whose sample levels are taken from its per-level lists
(`pigIF_thkperi_UniqueAges`), and SPARC:89's traces for the trace counts.

In [ ]:
hum_code = v.mat("65", "files/code/CadaverVNMorphology_OutputMetrics.mat")
hsub = pd.DataFrame(dict(label=hum_code["sub_sam"], level=hum_code["level"], sex=hum_code["sex"]))
hsub["sub"] = hsub["label"].str[1:].str.split("-").str[0].astype(int)
hsub["sam"] = hsub["label"].str.split("-").str[1].astype(int)
hsub["level"] = hsub["level"].map({"C": "cervical", "A": "subdiaphragmatic"})
hsubj = v.excel("65", "files/subjects.xlsx")
hsamp = v.excel("65", "files/samples.xlsx")
age_h = {int(s[4:]): (99 if (">=90" in a or "90+" in a) else int(str(a).split()[0])) for s, a in zip(hsubj["subject_id"], hsubj["age"])}
hsub["age"] = hsub["sub"].map(age_h)  # "90+" as 99, as DataExtraction_v3.m does
loc = {(int(s[4:]), int(m[4:])): l for s, m, l in zip(hsamp["subject_id"], hsamp["sample_id"], hsamp["specimen anatomical location"])}
hsub["location"] = [loc[(a, b)].split(";")[0] for a, b in zip(hsub["sub"], hsub["sam"])]
hum = {}
for _, row in hsub.iterrows():
    hum[row["label"]] = v.human_sample(row["sub"], row["sam"])
hsub["d_nerve_here"] = [hum[l]["d_nerve"] for l in hsub["label"]]
hsub["d_nerve_code"] = hum_code["dNerve_um"]
hsub["n_fasc_here"] = [hum[l]["n_fasc"] for l in hsub["label"]]
hsub["n_fasc_code"] = hum_code["NFasc"]
hsub["fasc_max_diff"] = [np.abs(np.sort(hum[l]["d_fasc"]) - np.sort(np.atleast_1d(hum_code["dFasc_um"][i]))).max()
                         for i, l in enumerate(hsub["label"])]
hsub["um_per_px"] = [hum[l]["um_per_px"] for l in hsub["label"]]
hsub["unmatched_inner"] = [hum[l]["n_unmatched"] for l in hsub["label"]]
hsub["nerve_objects"] = [hum[l]["n_nerve_objects"] for l in hsub["label"]]
hsub["nerve_mask_inverted"] = [hum[l]["nerve_mask_inverted"] for l in hsub["label"]]
print(hsub.round(3).to_string())
print(pd.crosstab(hsub["level"], hsub["sex"]), "| subjects:", hsub["sub"].nunique())
print("max |nerve diameter difference| %.2e um, max |fascicle diameter difference| %.2e um" %
      ((hsub.d_nerve_here - hsub.d_nerve_code).abs().max(), hsub.fasc_max_diff.max()))
print("samples in SPARC:65 not among the 18:", sorted(set(f"C{a}-{b}" for a, b in loc) - set(hsub["label"])))

In [ ]:
pig_code = v.mat("64", "files/code/PigVNMorphology_OutputMetrics.mat")
psub = pd.DataFrame(dict(label=pig_code["sub_sam"], level=pig_code["level"], sex=pig_code["sex"]))
psub["sub"] = psub["label"].str[1:].str.split("-").str[0].astype(int)
psub["sam"] = psub["label"].str.split("-").str[1].astype(int)
psub["level"] = psub["level"].map({"C": "cervical", "A": "subdiaphragmatic"})
psubj = v.excel("64", "files/subjects.xlsx")
age_p = {int(s[4:]): float(a.split()[0]) for s, a in zip(psubj["subject_id"], psubj["age"])}
psub["age"] = psub["sub"].map(age_p)
psub["d_nerve"] = pig_code["dNerve_um"].astype(float)
psub["n_fasc"] = pig_code["NFasc"].astype(int)
pig_dfasc = {l: np.atleast_1d(pig_code["dFasc_um"][i]).astype(float) for i, l in enumerate(psub["label"])}
psub["n_dfasc"] = [len(pig_dfasc[l]) for l in psub["label"]]
print(psub.round(2).to_string())
print(pd.crosstab(psub["level"], psub["sex"]), "| subjects:", psub["sub"].nunique(),
      "| NFasc equals the count of diameters in every sample:", bool((psub.n_fasc == psub.n_dfasc).all()))
allpig = np.concatenate(list(pig_dfasc.values()))
print("pig MT fascicle diameters: n = %d, %.1f to %.1f um" % (len(allpig), allpig.min(), allpig.max()))

In [ ]:
pif = v.mat("89", "files/code/PigIFMorphology.mat")
def _flat(a):
    out = []
    for x in np.atleast_1d(a):
        if isinstance(x, np.ndarray) and x.dtype == object:
            out.extend(np.atleast_1d(y).astype(float) for y in x)
        else:
            out.append(np.atleast_1d(x).astype(float))
    return out
if_d, if_t = _flat(pif["pigIF_dfasc_notpeanut"]), _flat(pif["pigIF_thkperi"])
if_dall = _flat(pif["pigIF_dfasc"])
if_endo, if_peri = _flat(pif["pigIF_Aendoperc"]), _flat(pif["pigIF_Aperiperc"])
if_nerve = _flat(pif["pigIF_dnerve"])
lvl_lists = [np.concatenate([np.atleast_1d(a).astype(float) for a in pif["pigIF_thkperi_UniqueAges"][k]]) for k in range(2)]
if_level = []
for t in if_t:
    inl = [np.isin(np.round(t, 10), np.round(L, 10)).all() for L in lvl_lists]
    assert sum(inl) == 1
    if_level.append("cervical" if inl[0] else "subdiaphragmatic")
idx = np.atleast_2d(pif["pigIF_sub_sam_indices"])
subs = np.atleast_1d(pif["pigIF_sub_values"])
pigif = pd.DataFrame(dict(level=if_level, n_nonpeanut=[len(d) for d in if_d], n_inner=[len(d) for d in if_dall],
                          d_nerve=[float(x[0]) for x in if_nerve], endo_pct=[float(x[0]) for x in if_endo],
                          peri_pct=[float(x[0]) for x in if_peri]))
print("PigIFMorphology.mat, per sample in file order (subjects", list(subs), "):")
print(pigif.round(2).to_string())
print("non-peanut fascicles: %d (cervical %d, subdiaphragmatic %d); file's own fit has dfe = %d, R2 = %.4f" %
      (pigif.n_nonpeanut.sum(), pigif[pigif.level == "cervical"].n_nonpeanut.sum(),
       pigif[pigif.level == "subdiaphragmatic"].n_nonpeanut.sum(), pif["pigIF_thkperi_gof"]["dfe"], pif["pigIF_thkperi_gof"]["rsquare"]))
pigif_traces = {}
for p in sorted(v.files("89")):
    if p.endswith("PeriInMask.tif"):
        sub, sam = p.split("/")[2][4:], p.split("/")[3][4:]
        pigif_traces[f"P{sub}-{sam}"] = v.pig_if_counts(int(sub), int(sam))
psamp89 = v.excel("89", "files/samples.xlsx")
loc89 = {f"P{s[4:]}-{m[4:]}": l for s, m, l in zip(psamp89["subject_id"], psamp89["sample_id"], psamp89["specimen anatomical location"])}
pit = pd.DataFrame(pigif_traces).T
pit["level"] = ["cervical" if "cervical" in loc89[k] else "subdiaphragmatic" for k in pit.index]
print(pit.to_string())
print("max rss %.2f GB, %.0f s" % (resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, time.time() - T0))

### Tables used by the claims

One row per sample for nerve-level measures and one per fascicle for fascicle diameters.
Rat statistics use one fascicle per sample (Methods, "Imaging and Image Analysis",
paragraph 2: "for all statistics, we only used the largest fascicle in each rat sample,
using the outer perineurial boundary as the nerve boundary"). The largest fascicle is
taken here by endoneurial area. `DataPlotting_v4.m` takes the largest outer perineurium
for the nerve, but for the fascicle diameter the first inner trace in its order
(`fasc_ind == 1`), which in R22-1 and R23-1 is not the largest fascicle; the code's values
are shown as a variant wherever they differ.

In [ ]:
rows = []
for _, r0 in smp.iterrows():
    r = rat[r0["label"]]
    lf = v.largest_fascicle(r)
    pv = v.rat_plot_values(r)
    rows.append(dict(species="rat", label=r0["label"], subject=f"R{r0['sub']}", level=r0["level"], sex=r0["sex"][0],
                     age=r0["age_days"], d_nerve=lf["d_nerve"], d_nerve_code=pv["d_nerve"], d_nerve_ct=r["d_nerve"],
                     d_fasc_largest=lf["d_fasc"], d_fasc_code=pv["d_fasc"], n_fasc=r["n_fasc"],
                     endo_area=lf["endo_area"], nerve_area=lf["outer_area"],
                     endo_area_all=r["endo_area"], peri_area_all=r["peri_area"], nerve_area_ct=r["nerve_area"],
                     endo_pct=100 * lf["endo_area"] / lf["outer_area"], peri_pct=100 * (lf["outer_area"] - lf["endo_area"]) / lf["outer_area"],
                     mean_fasc=lf["d_fasc"], fasc_range=0.0))
for _, h0 in hsub.iterrows():
    r = hum[h0["label"]]
    rows.append(dict(species="human", label=h0["label"], subject=f"C{h0['sub']}", level=h0["level"], sex=h0["sex"], age=h0["age"],
                     d_nerve=r["d_nerve"], n_fasc=r["n_fasc"], endo_area=r["endo_area"], nerve_area=r["nerve_area"],
                     endo_pct=100 * r["endo_area"] / r["nerve_area"], peri_pct=100 * r["peri_area"] / r["nerve_area"],
                     mean_fasc=r["d_fasc"].mean(), fasc_range=np.ptp(r["d_fasc"])))
for _, p0 in psub.iterrows():
    d = pig_dfasc[p0["label"]]
    a_endo = (np.pi * (d / 2) ** 2).sum()
    a_nerve = np.pi * (p0["d_nerve"] / 2) ** 2
    rows.append(dict(species="pig", label=p0["label"], subject=f"P{p0['sub']}", level=p0["level"], sex=p0["sex"], age=p0["age"],
                     d_nerve=p0["d_nerve"], n_fasc=p0["n_fasc"], endo_area=a_endo, nerve_area=a_nerve,
                     endo_pct=100 * a_endo / a_nerve, mean_fasc=d.mean(), fasc_range=np.ptp(d)))
S = pd.DataFrame(rows)
S["d_nerve_mm"] = S["d_nerve"] / 1000
fr = []
for _, s in S.iterrows():
    if s["species"] == "rat":
        ds_ = [s["d_fasc_largest"]]
    elif s["species"] == "human":
        ds_ = hum[s["label"]]["d_fasc"]
    else:
        ds_ = pig_dfasc[s["label"]]
    for d in ds_:
        fr.append(dict(species=s["species"], subject=s["subject"], sample=s["label"], level=s["level"], d_fasc=d,
                       d_fasc_code=s["d_fasc_code"] if s["species"] == "rat" else d))
FA = pd.DataFrame(fr)
FA["log_d"] = np.log10(FA["d_fasc"])
FA["log_d_code"] = np.log10(FA["d_fasc_code"])
print(S.groupby(["species", "level"]).agg(samples=("label", "size"), subjects=("subject", "nunique"),
      d_nerve_um=("d_nerve", "mean"), d_nerve_sd=("d_nerve", "std"), n_fasc=("n_fasc", "mean")).round(2))
print(FA.groupby(["species", "level"]).agg(fascicles=("d_fasc", "size"), mean_um=("d_fasc", "mean"), sd_um=("d_fasc", "std"),
      min_um=("d_fasc", "min"), max_um=("d_fasc", "max")).round(2))

def msd(x):
    return "%.4g +/- %.3g (n = %d)" % (np.mean(x), np.std(x, ddof=1), len(x))

def by(sp, lv, col="d_nerve", df=None):
    df = S if df is None else df
    return df[(df.species == sp) & (df.level == lv)][col].values

## rat-sample-set (link)

> The dataset holds the rat samples the paper analysed, segmented traces of nine left
> cervical and nine anterior subdiaphragmatic vagus nerve samples from Sprague-Dawley rats,
> 4F and 5M at each level, from 10 rats (5M/5F).

In [ ]:
print(pd.crosstab(smp["level"], smp["sex"]))
print("rats:", smp["sub"].nunique(), "| strain:", subjects["strain"].unique(), "| species:", subjects["species"].unique())
ages = subjects["age"].str.replace(" days", "").astype(int)
print("ages %d to %d days; body mass %.1f +/- %.1f g (SD, n = %d)" % (ages.min(), ages.max(), subjects["body mass"].mean(),
      subjects["body mass"].std(), len(subjects)))
print("cervical side:", smp[smp.level == "cervical"]["specimen anatomical location"].str.split(";").str[0].unique())
print("subdiaphragmatic location:", smp[smp.level == "subdiaphragmatic"]["specimen anatomical location"].str.split(";").str[0].unique())
print("R18 samples:", list(smp[smp["sub"] == 18]["label"] + " " + smp[smp["sub"] == 18]["level"]),
      "| R26 samples:", list(smp[smp["sub"] == 26]["label"] + " " + smp[smp["sub"] == 26]["level"]))
print("code sample lists (RCervicalLeft*, RAbdominalAnterior*):")
for f in ["RCervicalLeftMale", "RCervicalLeftFemale", "RAbdominalAnteriorMale", "RAbdominalAnteriorFemale",
          "RCervicalRightMale", "RAbdominalPosteriorFemale"]:
    t = v.excel("60", f"files/code/{f}.xlsx", header=None)
    print("  ", f, list(t[0]) if len(t) else [])

**Result: reproduced.** The dataset holds nine cervical and nine subdiaphragmatic rat
samples, 4F and 5M at each level, from 10 Sprague-Dawley rats (5M, 5F) aged 75 to 268 days
with body mass 381.1 +/- 118.4 g, as the Methods give. The cervical samples are all left;
the subdiaphragmatic samples are described as "subdiaphragmatic vagus nerve" without the
side, which the readme gives as anterior. R18 has only a cervical sample (R18-1) and R26
only a subdiaphragmatic one. No right cervical or posterior samples are deposited, and the
code's lists for them are empty. The readme says 65 to 268 days; subjects.xlsx gives 75.

## rat-nerve-diameter

> Rat vagus nerves have effective diameters of 0.260 +/- 0.025 mm at the cervical level and
> 0.135 +/- 0.015 mm at the subdiaphragmatic level.

In [ ]:
R = S[S.species == "rat"].copy()
print(R[["label", "level", "n_fasc", "d_nerve", "d_nerve_code", "d_nerve_ct"]].round(2).to_string())
for lv in ["cervical", "subdiaphragmatic"]:
    x = by("rat", lv) / 1000
    print(lv, "largest fascicle's outer perineurium: mean %.4f mm, SD %.4f mm, range %.4f to %.4f" % (x.mean(), x.std(ddof=1), x.min(), x.max()))
xc = by("rat", "subdiaphragmatic", "d_nerve_ct") / 1000
print("variant, connective tissue boundary for the multifascicular samples: mean %.4f, SD %.4f mm" % (xc.mean(), xc.std(ddof=1)))
r17 = v.rat_sample(17, 1)
nb17, _ = v.bwboundaries(v.tif("60", v._prefix("60", 17, 1) + "NerveMask.tif"))
print("R17-1 has one fascicle but a NerveMask: its diameter %.2f um (not used by the code)" % v.eff_d(v.polyarea(nb17[0]) * r17["um_per_px"] ** 2))

**Result: reproduced.** Cervical 0.2577 mm (SD 0.0249) and subdiaphragmatic 0.1352 mm
(SD 0.0148), over nine samples each, against 0.260 +/- 0.025 and 0.135 +/- 0.015; both means
and both SDs are within tolerance, and the values equal those of the paper's code. With the
connective tissue boundary for the three multifascicular samples the subdiaphragmatic mean
is 0.1748 mm (SD 0.0660).

## rat-subdiaphragmatic-multifascicular

> Rat vagus nerves are generally monofascicular; three of the nine rat subdiaphragmatic
> samples (R16-1, R22-1 and R23-1) are multifascicular.

In [ ]:
print(R[["label", "level", "n_fasc"]].to_string())
multi = sorted(R[(R.level == "subdiaphragmatic") & (R.n_fasc > 1)]["label"])
print("cervical with one fascicle:", int((R[R.level == "cervical"].n_fasc == 1).sum()), "of 9 | multifascicular subdiaphragmatic:", multi)

**Result: reproduced.** All nine cervical samples have one inner perineurium trace; three
subdiaphragmatic samples have more, R16-1 (2), R22-1 (3) and R23-1 (2), the three the
paper names.

## rat-multifascicular-proportions

> Counting all segmented fascicles and the surrounding tissue, the epineurium, perineurium
> and endoneurium proportions are 58.0, 3.8 and 38.2% for R16-1, 55.1, 4.2 and 40.7% for
> R23-1, and 63.4, 3.6 and 33.0% for R22-1.

In [ ]:
exp = {"R16-1": (58.0, 3.8, 38.2), "R23-1": (55.1, 4.2, 40.7), "R22-1": (63.4, 3.6, 33.0)}
ok = 0
for lab, e in exp.items():
    r = rat[lab]
    endo = 100 * r["endo_area"] / r["nerve_area"]
    peri = 100 * r["peri_area"] / r["nerve_area"]
    epi = 100 - endo - peri
    got = (epi, peri, endo)
    within = [abs(g - x) <= 0.1 * x + 1e-9 for g, x in zip(got, e)]
    ok += sum(within)
    print(lab, "epineurium %.1f, perineurium %.2f, endoneurium %.1f %% | expected %s | within 10%%: %s" % (epi, peri, endo, e, within))
print("values within 10%%: %d of 9" % ok)

**Result: reproduced.** All nine proportions are within 10% of the legend's values: R16-1
58.0, 3.8 and 38.2%, R23-1 55.1, 4.2 and 40.7%, R22-1 63.4, 3.6 and 33.0% (epineurium,
perineurium, endoneurium).

## rat-fascicle-diameter

> The largest rat fascicle has an effective diameter of 0.249 +/- 0.024 mm at the cervical
> level and 0.121 +/- 0.032 mm at the subdiaphragmatic level.

In [ ]:
print(R[["label", "level", "n_fasc", "d_fasc_largest", "d_fasc_code"]].round(2).to_string())
for lv in ["cervical", "subdiaphragmatic"]:
    x, xc_ = by("rat", lv, "d_fasc_largest") / 1000, by("rat", lv, "d_fasc_code") / 1000
    print(lv, "largest: mean %.4f SD %.4f mm, geometric mean %.4f | code's first trace: mean %.4f SD %.4f mm" %
          (x.mean(), x.std(ddof=1), stats.gmean(x), xc_.mean(), xc_.std(ddof=1)))

**Result: reproduced.** The largest fascicle has a mean effective diameter of 0.2486 mm
cervically (SD 0.0243) and 0.1291 mm subdiaphragmatically (SD 0.0140), within 0.249 +/- 0.024
and 0.121 +/- 0.032. The paper's code takes the first inner trace in its order, which is not
the largest fascicle in R22-1 (39.35 against 105.76 um) and R23-1 (142.15 against 147.47 um);
with its values the subdiaphragmatic mean is 0.1211 mm (SD 0.0323), the paper's numbers.
Geometric means are 0.2475 and 0.1284 mm.

## nerve-diameter-anova

> Effective nerve diameter differs across species and across nerve levels, with no
> detected interaction of species and level.

The 54 samples: rat from this dataset (largest fascicle's outer perineurium), human from
SPARC:65's traces, pig MT from SPARC:64's output file. Subjects are coded per species.

The Methods fit the models in JMP Pro 14.3.0, whose REML default allows negative variance
components ("Unbounded Variance Components"), and test effects with Kenward-Roger F tests.
The main fit here is REML with unbounded variance components (`vagus.reml_unbounded`),
tested by Wald F with Satterthwaite denominator degrees of freedom (`p_satt`), the
approximation Kenward-Roger reduces to when its bias correction is small; Tukey tests use
the Satterthwaite degrees of freedom of each difference. p with the residual degrees of
freedom and with subjects minus fixed parameters are shown too, and the statsmodels REML
fit, whose subject variance cannot go below zero, is a variant.

In [ ]:
tab_nd, vc_nd, fit_nd = v.reml_unbounded(S, "d_nerve_mm", "subject")
print("unbounded REML: variance components", {k: round(x, 5) for k, x in vc_nd.items()}, "| converged:", fit_nd["converged"])
print(tab_nd.round(5).to_string())
res_nd, tab_nd_b = v.mixed_anova(S, "d_nerve_mm")
print("variant, statsmodels REML (bounded): subject variance %.5f" % res_nd.cov_re.iloc[0, 0])
print(tab_nd_b.round(5).to_string())
print("paper: species F = 577.1 p < 0.0001; level F = 11.01 p = 0.0030; interaction F = 2.445 p = 0.1085")
Sc = S.copy(); Sc["d_nerve_mm"] = Sc["d_nerve_code"].fillna(Sc["d_nerve"]) / 1000
print("rat values as the code takes them are the same samples' values:", bool(np.allclose(Sc.d_nerve_mm, S.d_nerve_mm)))
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
print("variant, ordinary two-way ANOVA ignoring subject (type II):")
print(anova_lm(ols("d_nerve_mm ~ C(species) * C(level)", S).fit(), typ=2).round(4))

**Result: reproduced (resting on an assumption).** With unbounded variance components (the
subject variance is estimated at -0.052, against a residual 0.127), species F = 599.6
(p < 0.0001), level F = 11.05 (Satterthwaite p = 0.0029) and the interaction F = 2.455
(p = 0.107), against the paper's 577.1, 11.01 (p = 0.0030) and 2.445 (p = 0.1085): species
and level are significant and the interaction is not. With the subject variance held at
zero or above (statsmodels), the interaction is significant (F = 3.71, p = 0.024), which
would make the claim only partially reproduced; the outcome rests on JMP's default.

## human-pig-nerve-diameter-similar

> Human and pig vagus nerves have similar effective diameters, 1.9 +/- 0.5 against
> 2.2 +/- 0.2 mm cervically and 1.7 +/- 0.3 against 1.6 +/- 0.2 mm subdiaphragmatically, with
> no significant difference by Tukey's test.

In [ ]:
for sp in ["human", "pig"]:
    for lv in ["cervical", "subdiaphragmatic"]:
        print(sp, lv, msd(by(sp, lv, "d_nerve_mm")), "mm")
fit_nd_t = dict(fit_nd, satt=True)
tk_sp, means_sp = v.tukey(fit_nd_t, S, [("human", "pig"), ("rat", "human"), ("rat", "pig")], "a")
print("species least squares means (mm):", {k: round(x, 4) for k, x in means_sp.items()})
print(tk_sp.round(5).to_string())
cells = [(("human", lv), ("pig", lv)) for lv in ["cervical", "subdiaphragmatic"]] + \
        [(("rat", lv), (o, lv)) for lv in ["cervical", "subdiaphragmatic"] for o in ["human", "pig"]]
tk_cell, means_cell = v.tukey(fit_nd_t, S, cells, "cell")
tk_sp["p_subject_df"] = v.tukey(dict(fit_nd, dfres=fit_nd["dfsub"]), S, [("human", "pig"), ("rat", "human"), ("rat", "pig")], "a")[0]["p_tukey"]
print(tk_sp.round(5).to_string())
tk_sp_b, _ = v.tukey(v.sm_fit(res_nd), S, [("human", "pig"), ("rat", "human"), ("rat", "pig")], "a")
print("variant, Tukey on the bounded fit:")
print(tk_sp_b.round(5).to_string())
print("variant, species by level cells:")
print(tk_cell.round(5).to_string())
hp = S[S.species.isin(["human", "pig"])]
d_eff = (hp[hp.species == "human"].d_nerve_mm.mean() - hp[hp.species == "pig"].d_nerve_mm.mean()) / hp.d_nerve_mm.std(ddof=1)
print("effect size, human minus pig over the pooled SD of the 36 samples: %.3f" % d_eff)

**Result: reproduced.** Human 1.922 +/- 0.460 and pig 2.195 +/- 0.195 mm cervically,
1.677 +/- 0.316 and 1.638 +/- 0.161 mm subdiaphragmatically, each within the paper's mean and
SD. Tukey's test of the species means gives human against pig p = 0.157 (paper 0.1830);
per level 0.135 and 0.988; with the bounded fit 0.343. The difference of the human and
pig means is 0.32 pooled SDs.

## rat-ten-times-smaller

> Rat vagus nerves are about ten times smaller in diameter than human and pig nerves,
> significantly smaller than each of the two larger species.

In [ ]:
print(tk_sp[tk_sp.pair.str.startswith("rat")].to_string(float_format=lambda x: "%.3g" % x))
for lv in ["cervical", "subdiaphragmatic"]:
    rm = by("rat", lv, "d_nerve_mm").mean()
    print(lv, "rat/human %.3f, rat/pig %.3f" % (rm / by("human", lv, "d_nerve_mm").mean(), rm / by("pig", lv, "d_nerve_mm").mean()))
print(tk_cell[tk_cell.pair.str.contains("'rat'")].to_string(float_format=lambda x: "%.3g" % x))

**Result: reproduced.** Rat against human and rat against pig both have Tukey p < 0.0001
(4.1e-14 and 2.6e-12), with rat smaller, and the ratios of the rat mean to the human and pig
means are 0.134 and 0.117 cervically and 0.081 and 0.082 subdiaphragmatically, all within
0.05 to 0.2. Per level, all four contrasts have p < 0.0001.

## nerve-cervical-larger

> Cervical vagus nerves are larger than subdiaphragmatic nerves, a trend clearest in pigs
> and rats.

In [ ]:
lsm = {}
for lv in ["cervical", "subdiaphragmatic"]:
    lsm[lv] = np.mean([means_cell[(sp, lv)] for sp in ["human", "pig", "rat"]])
print("level effect: F = %.3f, Satterthwaite p = %.4g (residual df %.4g, subject df %.4g) | least squares means, cervical %.4f mm, subdiaphragmatic %.4f mm" %
      (tab_nd.F[1], tab_nd.p_satt[1], tab_nd.p_resid_df[1], tab_nd.p_subject_df[1], lsm["cervical"], lsm["subdiaphragmatic"]))
rel = {}
for sp in ["human", "pig", "rat"]:
    c, a = by(sp, "cervical", "d_nerve_mm").mean(), by(sp, "subdiaphragmatic", "d_nerve_mm").mean()
    rel[sp] = (c - a) / c
    print(sp, "cervical %.4f, subdiaphragmatic %.4f mm, relative difference %.3f" % (c, a, rel[sp]))
print("pig and rat larger than human:", rel["pig"] > rel["human"] and rel["rat"] > rel["human"])
print("rat cervical mean above subdiaphragmatic:", by("rat", "cervical").mean() > by("rat", "subdiaphragmatic").mean())

**Result: reproduced (resting on an assumption).** The level effect has p = 0.0029, and
the cervical least squares mean (1.425 mm) exceeds the subdiaphragmatic (1.131 mm). The
relative difference of means is 0.127 in humans, 0.254 in pigs and 0.475 in rats, so larger
for pigs and rats than for humans, the claims file's reading of "clearest". The rat part
alone holds (0.258 against 0.135 mm).

## fascicle-number-human-pig

> Pig vagus nerves have almost ten times more fascicles than human nerves (47 +/- 10 and
> 38 +/- 7 against 6.7 +/- 4.5 and 4.9 +/- 4.0 at the cervical and subdiaphragmatic levels),
> the number differs by species and level with no detected interaction, and there are more
> fascicles cervically.

In [ ]:
HP = S[S.species.isin(["human", "pig"])].copy()
HP["n_fasc"] = HP["n_fasc"].astype(float)
for sp in ["pig", "human"]:
    for lv in ["cervical", "subdiaphragmatic"]:
        print(sp, lv, msd(by(sp, lv, "n_fasc")))
tab_nf, vc_nf, fit_nf = v.reml_unbounded(HP, "n_fasc", "subject")
print("unbounded REML: variance components", {k: round(x, 4) for k, x in vc_nf.items()}, "| converged:", fit_nf["converged"])
print(tab_nf.round(5).to_string())
res_nf, tab_nf_b = v.mixed_anova(HP, "n_fasc")
print("variant, statsmodels REML (bounded):")
print(tab_nf_b.round(5).to_string())
print("paper: species F = 213.8 p < 0.0001; level F = 5.486 p = 0.0300; interaction F = 2.481 p = 0.1322")
print("pig/human ratio of means: cervical %.2f, subdiaphragmatic %.2f" % (by("pig", "cervical", "n_fasc").mean() / by("human", "cervical", "n_fasc").mean(),
      by("pig", "subdiaphragmatic", "n_fasc").mean() / by("human", "subdiaphragmatic", "n_fasc").mean()))

**Result: reproduced.** Pig 46.89 +/- 10.3 and 38.22 +/- 7.08 fascicles, human 6.667 +/- 4.5
and 4.889 +/- 3.95, each within the paper's mean and SD. Species F = 216.7 (p < 0.0001),
level F = 5.79 (p = 0.026) and interaction F = 2.61 (p = 0.123), against 213.8, 5.486
(p = 0.0300) and 2.481 (p = 0.1322); more fascicles cervically in both species. Pigs have
7.0 and 7.8 times the human mean.

## fascicle-diameter-anova

> Effective fascicle diameter differs across species and nerve levels, with a significant
> species by level interaction.

log10 of each fascicle's effective diameter: every inner perineurium trace of the human
(104) and pig MT (766) samples, and the largest fascicle of each rat sample (18).

In [ ]:
tab_fd, vc_fd, fit_fd = v.reml_unbounded(FA, "log_d", "subject", nested="sample")
print("unbounded REML: variance components", {k: round(x, 5) for k, x in vc_fd.items()}, "| converged:", fit_fd["converged"])
print(tab_fd.round(5).to_string())
print("paper: species F = 8.989 p = 0.0015; level F = 51.88 p < 0.0001; interaction F = 5.273 p = 0.0121")
tab_fdc, vc_fdc, fit_fdc = v.reml_unbounded(FA, "log_d_code", "subject", nested="sample", x0=fit_fd["theta"])
print("variant, rat fascicles as the code takes them: converged", fit_fdc["converged"])
print(tab_fdc.round(5).to_string())
res_fd, tab_fd_b = v.mixed_anova(FA, "log_d", sample="sample")
print("variant, statsmodels REML (bounded):")
print(tab_fd_b.round(5).to_string())

**Result: reproduced.** On log10 fascicle diameter, species F = 9.74 (p = 0.0011), level
F = 48.0 (p < 0.0001) and interaction F = 5.46 (p = 0.0085), all below 0.05 (paper 8.989,
p = 0.0015; 51.88; 5.273, p = 0.0121). With the rat fascicles as the paper's code takes them,
species F = 9.13 (p = 0.0015), level F = 53.6 and interaction F = 5.79 (p = 0.0069).

## cervical-fascicle-diameter-species

> At the cervical level human fascicles (0.355 +/- 0.226 mm) are larger than pig fascicles
> (0.155 +/- 0.058 mm), while rat fascicles are comparable to both.

In [ ]:
for sp in ["human", "pig", "rat"]:
    x = by(sp, "cervical", "d_fasc", FA) / 1000
    pm = FA[(FA.species == sp) & (FA.level == "cervical")].groupby("sample").d_fasc.mean() / 1000
    print(sp, "cervical, over fascicles", msd(x), "mm | mean of sample means %.4f +/- %.4f (n = %d)" % (pm.mean(), pm.std(ddof=1), len(pm)))
pairs_c = [(("human", "cervical"), ("pig", "cervical")), (("rat", "cervical"), ("human", "cervical")), (("rat", "cervical"), ("pig", "cervical"))]
pairs_a = [(("human", "subdiaphragmatic"), ("pig", "subdiaphragmatic")), (("human", "subdiaphragmatic"), ("rat", "subdiaphragmatic")),
           (("pig", "subdiaphragmatic"), ("rat", "subdiaphragmatic"))]
pairs_l = [((sp, "cervical"), (sp, "subdiaphragmatic")) for sp in ["human", "pig", "rat"]]
tk_fd, means_fd = v.tukey(dict(fit_fd, satt=True), FA, pairs_c + pairs_a + pairs_l, "cell")
tk_fd["p_subject_df"] = v.tukey(dict(fit_fd, dfres=fit_fd["dfsub"]), FA, pairs_c + pairs_a + pairs_l, "cell")[0]["p_tukey"]
tk_fd["p_resid_df"] = v.tukey(fit_fd, FA, pairs_c + pairs_a + pairs_l, "cell")[0]["p_tukey"]
tk_fd["p_code_rat"] = v.tukey(dict(fit_fdc, satt=True), FA, pairs_c + pairs_a + pairs_l, "cell")[0]["p_tukey"]
tk_fd["p_code_rat_subject_df"] = v.tukey(dict(fit_fdc, dfres=fit_fdc["dfsub"]), FA, pairs_c + pairs_a + pairs_l, "cell")[0]["p_tukey"]
tk_fd["p_tukey_bounded"] = v.tukey(v.sm_fit(res_fd), FA, pairs_c + pairs_a + pairs_l, "cell")[0]["p_tukey"]
print("cell least squares means, back-transformed (mm):", {f"{k[0]} {k[1][:4]}": round(10 ** x / 1000, 4) for k, x in means_fd.items()})
print("paper: human vs pig p = 0.0002; rat vs human p = 0.7270; rat vs pig p = 0.1707")
print(tk_fd.iloc[:3].round(5).to_string())

**Result: reproduced.** Over fascicles, human 0.3548 +/- 0.226 mm (60 fascicles) and pig
0.1545 +/- 0.0579 mm (422), within the paper's values; human against pig Tukey p = 0.00024
with human larger (paper 0.0002), rat against human p = 0.716 (0.7270) and rat against pig
p = 0.137 (0.1707). With the subject-based degrees of freedom the three p are 0.00009, 0.716
and 0.155. The means of sample means (0.419 and 0.156 mm) are also within the SDs.

## subdiaphragmatic-fascicle-diameter-comparable

> At the subdiaphragmatic level fascicle diameters are comparable across species (human
> 0.163 +/- 0.146, pig 0.110 +/- 0.044, rat 0.121 +/- 0.032 mm), although human fascicles
> trend larger than pig fascicles.

In [ ]:
for sp in ["human", "pig", "rat"]:
    print(sp, "subdiaphragmatic, over fascicles", msd(by(sp, "subdiaphragmatic", "d_fasc", FA) / 1000), "mm")
print("paper: human vs pig p = 0.3049; human vs rat p = 0.8960; pig vs rat p = 0.9947")
print(tk_fd.iloc[3:6].round(5).to_string())
print("effect sizes, differences of log10 least squares means:", tk_fd.iloc[3:6]["diff"].round(4).tolist())

**Result: reproduced.** Human 0.1635 +/- 0.146 mm and pig 0.1101 +/- 0.0438 mm, within the
paper's values; no subdiaphragmatic pair differs (human against pig p = 0.283, human against
rat 0.995, pig against rat 0.883; paper 0.3049, 0.8960, 0.9947). The log10 differences are
0.146, 0.049 and -0.097, the largest human against pig, as the paper's "trended larger".
With the code's rat values, the two rat pairs give 0.896 and 0.995, the paper's values.

## fascicles-larger-cervical

> All three species have larger fascicles at the cervical than at the subdiaphragmatic
> level.

In [ ]:
print("paper: humans p < 0.0001; pigs p = 0.0009; rats p = 0.0234")
print(tk_fd.iloc[6:9].round(6).to_string())
RR = FA[FA.species == "rat"].copy()
import statsmodels.formula.api as smf
for col in ["log_d", "log_d_code"]:
    rr = smf.mixedlm(f"{col} ~ C(level)", RR, groups=RR["subject"]).fit(reml=True)
    print("variant, rat-only mixed model on %s: level coefficient %.4f, p = %.3g" % (col, rr.params.iloc[1], rr.pvalues.iloc[1]))

**Result: reproduced (resting on an assumption).** Cervical fascicles are larger in all
three species: human p = 0.00001, pig p = 0.048 and rat p = 0.044 with Satterthwaite degrees
of freedom (paper < 0.0001, 0.0009, 0.0234). The pig and rat p depend on the degrees of
freedom: with the residual ones pig 0.00001 and rat 0.043, with subjects minus parameters pig
0.00036 and rat 0.068, which would leave two of three species. With the paper's code's rat
values the rat p is 0.0091. The rat-only variant gives p < 0.0001 for both rat readings. The
outcome rests on the choice of denominator degrees of freedom, which JMP fixes by
Kenward-Roger and which is not available here.

## rat-diameter-like-fascicle

> The monofascicular rat nerves have diameters close to that of a single human or pig
> fascicle.

In [ ]:
for lv in ["cervical", "subdiaphragmatic"]:
    rn = by("rat", lv, "d_nerve").mean()
    hf, pf = by("human", lv, "d_fasc", FA), by("pig", lv, "d_fasc", FA)
    lo, hi = sorted([hf.mean(), pf.mean()])
    rng = (min(hf.min(), pf.min()), max(hf.max(), pf.max()))
    print("%s: rat nerve %.1f um | human fascicle mean %.1f, pig %.1f | (a) between: %s | (b) range %.1f to %.1f: %s" %
          (lv, rn, hf.mean(), pf.mean(), lo <= rn <= hi, rng[0], rng[1], rng[0] <= rn <= rng[1]))

**Result: reproduced.** Under reading (a), the rat mean nerve diameter lies between the pig
and human mean fascicle diameters at both levels (257.7 between 154.5 and 354.8 um; 135.2
between 110.1 and 163.5 um), and under reading (b) within the range of their fascicles.

## human-pig-tissue-proportions

> Endoneurium is a larger share of the human nerve cervically (32 +/- 7.9%) than
> subdiaphragmatically (8.8 +/- 5.3%), while in pigs it is more even across levels
> (27 +/- 5.3% and 20 +/- 4.0%, MT); perineurium follows the same trends (human
> 8.9 +/- 2.5% and 3.4 +/- 1.6%, pig 5.1 +/- 1.8% and 4.0 +/- 0.5%).

Human from the traces (endoneurium within inner traces, perineurium between outer and
inner traces, over the nerve trace); pig MT endoneurium from the per-fascicle diameters of
SPARC:64 (the sum of the fascicles' circle areas over the nerve's circle area, the same
areas as the traces' since each diameter is that of the trace's area); pig perineurium from
SPARC:89's per-sample IF proportions.

In [ ]:
exp_tp = [("human", "cervical", "endo_pct", 32, 7.9), ("human", "subdiaphragmatic", "endo_pct", 8.8, 5.3),
          ("pig", "cervical", "endo_pct", 27, 5.3), ("pig", "subdiaphragmatic", "endo_pct", 20, 4.0),
          ("human", "cervical", "peri_pct", 8.9, 2.5), ("human", "subdiaphragmatic", "peri_pct", 3.4, 1.6),
          ("pigIF", "cervical", "peri_pct", 5.1, 1.8), ("pigIF", "subdiaphragmatic", "peri_pct", 4.0, 0.5)]
meas_tp = []
for sp, lv, col, e, tol in exp_tp:
    x = pigif[pigif.level == lv][col].values if sp == "pigIF" else by(sp, lv, col)
    meas_tp.append(x.mean())
    print("%-6s %-16s %s: %s %% | expected %.1f +/- %.1f | within: %s" % (sp, lv, col, msd(x), e, tol, abs(x.mean() - e) <= tol))
for lv in ["cervical", "subdiaphragmatic"]:
    print("pig IF endoneurium, %s: %s %%" % (lv, msd(pigif[pigif.level == lv].endo_pct.values)))

**Result: reproduced.** Endoneurium: human 32.23 +/- 7.9% cervically and 8.83 +/- 5.3%
subdiaphragmatically, pig MT 26.63 +/- 5.27% and 20.29 +/- 3.98%. Perineurium: human
8.87 +/- 2.47% and 3.42 +/- 1.59%, pig IF 5.07 +/- 1.75% and 4.00 +/- 0.52%. All eight means are
within the stated SDs. The pig IF endoneurium (24.9 and 18.1%) is about 2 points lower than
MT, as the paper says.

## rat-level-ratios

> In rats the ratio of mean subdiaphragmatic to mean cervical endoneurial area is 0.32 and
> the ratio of mean nerve diameters is 0.68.

In [ ]:
ra = R.set_index("label")
def ratio(col_a, col_c):
    return ra[ra.level == "subdiaphragmatic"][col_a].mean() / ra[ra.level == "cervical"][col_c].mean()
rd_a = ratio("endo_area", "endo_area"), ratio("d_nerve", "d_nerve")
rd_b = ratio("endo_area_all", "endo_area_all"), ratio("d_nerve_ct", "d_nerve_ct")
print("(a) largest fascicle: endoneurial area ratio %.3f, nerve diameter ratio %.3f" % rd_a)
print("(b) all fascicles and connective tissue boundary: endoneurial area ratio %.3f, nerve diameter ratio %.3f" % rd_b)
print("expected 0.32 +/- 0.032 and 0.68 +/- 0.068; the Results' means give 0.135/0.260 = %.3f" % (0.135 / 0.260))
print("within: (a) %s, %s | (b) %s, %s" % (abs(rd_a[0] - 0.32) <= 0.032, abs(rd_a[1] - 0.68) <= 0.068, abs(rd_b[0] - 0.32) <= 0.032, abs(rd_b[1] - 0.68) <= 0.068))
print("other ratios: mean nerve area (a) %.3f, (b) %.3f; mean fascicle diameter %.3f" % (
    ratio("nerve_area", "nerve_area"), ratio("nerve_area_ct", "nerve_area_ct"), ratio("d_fasc_largest", "d_fasc_largest")))

**Result: reproduced.** Under reading (b), all fascicles for the endoneurial area and the
connective tissue boundary for the nerve diameter of the three multifascicular samples, the
ratios are 0.317 and 0.678, within 10% of 0.32 and 0.68. Under reading (a), the largest
fascicle as in the statistics, they are 0.270 and 0.525 (the Results' means give 0.519), both
outside. So the Discussion's rat ratios use a different boundary than the Results' rat
statistics.

## human-pig-level-ratios

> The ratio of mean subdiaphragmatic to mean cervical endoneurial area is 0.20 in humans and
> 0.42 in pigs, and the ratio of mean nerve diameters is 0.87 and 0.75.

In [ ]:
meas_hpr = []
for col, sp, e in [("endo_area", "human", 0.20), ("endo_area", "pig", 0.42), ("d_nerve", "human", 0.87), ("d_nerve", "pig", 0.75)]:
    rr_ = by(sp, "subdiaphragmatic", col).mean() / by(sp, "cervical", col).mean()
    meas_hpr.append(rr_)
    print("%s %s ratio %.3f | expected %.2f +/- %.3f | within: %s" % (sp, col, rr_, e, 0.1 * e, abs(rr_ - e) <= 0.1 * e + 1e-12))

**Result: reproduced.** Endoneurial area ratios 0.198 (human) and 0.421 (pig), nerve
diameter ratios 0.873 and 0.746, each within 10% of 0.20, 0.42, 0.87 and 0.75.

## human-variability

> Nerve and fascicle diameters are more consistent across samples in pigs (and rats) than
> in humans.

In [ ]:
def cv(x):
    return np.std(x, ddof=1) / np.mean(x)
rows_cv = []
for lv in ["cervical", "subdiaphragmatic"]:
    for sp in ["human", "pig", "rat"]:
        rows_cv.append(dict(level=lv, species=sp, cv_nerve=cv(by(sp, lv, "d_nerve")), cv_mean_fasc=cv(by(sp, lv, "mean_fasc")),
                            cv_fasc_range=cv(by(sp, lv, "fasc_range")) if sp != "rat" else np.nan))
CV = pd.DataFrame(rows_cv)
print(CV.round(3).to_string())
for lv in ["cervical", "subdiaphragmatic"]:
    c = CV[CV.level == lv].set_index("species")
    a_hp = c.loc["human", "cv_nerve"] > c.loc["pig", "cv_nerve"] and c.loc["human", "cv_mean_fasc"] > c.loc["pig", "cv_mean_fasc"]
    a_hr = c.loc["human", "cv_nerve"] > c.loc["rat", "cv_nerve"] and c.loc["human", "cv_mean_fasc"] > c.loc["rat", "cv_mean_fasc"]
    b_hp = c.loc["human", "cv_nerve"] > c.loc["pig", "cv_nerve"] and c.loc["human", "cv_fasc_range"] > c.loc["pig", "cv_fasc_range"]
    print("%s: (a) human > pig %s, human > rat %s | (b) human > pig %s" % (lv, a_hp, a_hr, b_hp))

**Result: reproduced (resting on an assumption).** The coefficient of variation of nerve diameter is 0.239 (human),
0.089 (pig) and 0.097 (rat) cervically and 0.189, 0.099 and 0.110 subdiaphragmatically; of the
per-sample mean fascicle diameter 0.314, 0.077 and 0.098, and 0.662, 0.141 and 0.109. Human
exceeds pig and rat on both measures at both levels under reading (a), and pig under reading
(b) (fascicle range CV 0.500 against 0.166, and 0.853 against 0.192).

## literature-comparison-human-pig and rat-literature-comparison

> Human cervical fascicle numbers and endoneurial areas match prior reports while the nerve
> areas and diameters are generally smaller; human subdiaphragmatic sizes match prior
> reports; pig fascicle numbers match literature, with slightly smaller mean diameters.

> Rat cervical nerve diameters and areas agree with published data, while the rat
> subdiaphragmatic nerves are slightly larger than prior data.

The literature values are those of Supplement 4 ("Summary statistics and literature
review", Data_Sheet_2.xlsx), which the claims file's reader did not have; it is read from
the supplementary files of the version of record in Europe PMC (PMC7672126). Its
summary columns for this paper's own data are checked against the values measured here.
As the criterion says, each literature mean is judged against this paper's mean +/- SD
(Figure 7).

In [ ]:
LIT = v.supplement4()
print("Supplement 4 rows:", len(LIT), "| sections:", list(LIT.section.unique()))
sec = {"area": "Total nerve cross-sectional area (mm^2)", "diam": "Effective nerve diameter (mm)",
       "nfasc": "Number of fascicles (unitless)", "endo": "Endoneurial cross-sectional area (mm^2)"}
lvmap = {"Cervical": "cervical", "Abdominal": "subdiaphragmatic"}
ours = {"area": lambda sp, lv: by(sp, lv, "nerve_area") / 1e6, "diam": lambda sp, lv: by(sp, lv, "d_nerve_mm"),
        "nfasc": lambda sp, lv: by(sp, lv, "n_fasc").astype(float), "endo": lambda sp, lv: by(sp, lv, "endo_area") / 1e6}
chk = []
for key, title in sec.items():
    for (sp, lv), g in LIT[LIT.section == title].groupby(["species", "level"]):
        if sp not in ("Human", "Pig", "Rat"):
            continue
        x = ours[key](sp.lower(), lvmap[lv])
        chk.append(dict(measure=key, species=sp, level=lv, supplement_mean=g.ours_mean.iloc[0], here_mean=x.mean(),
                        supplement_sd=g.ours_sd.iloc[0], here_sd=np.std(x, ddof=1)))
print(pd.DataFrame(chk).round(4).to_string())

def lit_table(key, sp, lv):
    g = LIT[(LIT.section == sec[key]) & (LIT.species == sp) & (LIT.level == lv) & LIT.lit_mean.notna()].copy()
    x = ours[key](sp.lower(), lvmap[lv])
    m, sd = x.mean(), np.std(x, ddof=1)
    g["here_mean"], g["here_sd"], g["here_min"], g["here_max"] = m, sd, x.min(), x.max()
    g["within_mean_sd"] = (g.lit_mean >= m - sd) & (g.lit_mean <= m + sd)
    g["here_below"] = m < g.lit_mean
    lo, hi = g.lit_mean - g.lit_var, g.lit_mean + g.lit_var
    g["ranges_overlap"] = np.where(g.lit_var.notna(), (x.min() <= hi) & (x.max() >= lo), np.nan)
    return g[["reference", "laterality", "sex", "n", "lit_mean", "lit_var", "var_kind", "here_mean", "here_sd", "here_min",
              "here_max", "within_mean_sd", "here_below", "ranges_overlap"]]

re_ = LIT[(LIT.section == sec["endo"]) & (LIT.species == "Rat")][["level", "ours_mean", "ours_min", "ours_max"]].drop_duplicates()
print("Supplement 4, rat endoneurial area (mm2):"); print(re_.round(4).to_string())
print("rat subdiaphragmatic endoneurial area over all fascicles here: mean %.4f, max %.4f mm2; ratio to cervical %.3f" % (
    by("rat", "subdiaphragmatic", "endo_area_all").mean() / 1e6, by("rat", "subdiaphragmatic", "endo_area_all").max() / 1e6,
    by("rat", "subdiaphragmatic", "endo_area_all").mean() / by("rat", "cervical", "endo_area").mean()))

Supplement 4's summaries of this paper's data equal the values measured here, with one
exception: its rat subdiaphragmatic endoneurial area is the area of all segmented fascicles
(mean 0.0155, maximum 0.0329 mm2, as here), although its note says the rat data are for the
largest fascicle (0.0132 mm2 here). Over all fascicles the ratio to the cervical area is
0.317, the Discussion's 0.32.

In [ ]:
parts = {}
for name, key, sp, lv in [("human cervical fascicles", "nfasc", "Human", "Cervical"), ("human cervical endoneurial area", "endo", "Human", "Cervical"),
                          ("human cervical nerve area", "area", "Human", "Cervical"), ("human cervical nerve diameter", "diam", "Human", "Cervical"),
                          ("human subdiaphragmatic nerve area", "area", "Human", "Abdominal"), ("human subdiaphragmatic nerve diameter", "diam", "Human", "Abdominal"),
                          ("human subdiaphragmatic fascicles", "nfasc", "Human", "Abdominal"),
                          ("pig cervical fascicles", "nfasc", "Pig", "Cervical"), ("pig subdiaphragmatic fascicles", "nfasc", "Pig", "Abdominal"),
                          ("pig cervical nerve diameter", "diam", "Pig", "Cervical"), ("pig subdiaphragmatic nerve diameter", "diam", "Pig", "Abdominal")]:
    t = lit_table(key, sp, lv)
    parts[name] = t
    print("=====", name)
    print(t.round(4).to_string())
ant = lambda t: t[t.laterality.astype(str).str.startswith("Ant")]
verdict = {
    "human cervical fascicles comparable": bool(parts["human cervical fascicles"].within_mean_sd.all()),
    "human cervical endoneurial areas comparable": bool(parts["human cervical endoneurial area"].within_mean_sd.all()),
    "human cervical nerve areas smaller": bool(parts["human cervical nerve area"].here_below.all()),
    "human cervical nerve diameters smaller": bool(parts["human cervical nerve diameter"].here_below.all()),
    "human cervical below 3.0 mm": bool(by("human", "cervical", "d_nerve_mm").mean() < 3.0),
    "human cervical ranges overlap (where a spread is given)": bool(pd.concat([parts["human cervical nerve area"], parts["human cervical nerve diameter"]]).ranges_overlap.dropna().astype(bool).all()),
    "human subdiaphragmatic anterior areas compare well": bool(ant(parts["human subdiaphragmatic nerve area"]).within_mean_sd.all()),
    "human subdiaphragmatic anterior diameters compare well": bool(ant(parts["human subdiaphragmatic nerve diameter"]).within_mean_sd.all()),
    "human subdiaphragmatic fewer fascicles than the posterior study": bool(parts["human subdiaphragmatic fascicles"].here_below.all()),
    "pig fascicles consistent": bool(parts["pig cervical fascicles"].within_mean_sd.all() and parts["pig subdiaphragmatic fascicles"].within_mean_sd.all()),
    "pig mean diameters smaller": bool(parts["pig cervical nerve diameter"].here_below.all() and parts["pig subdiaphragmatic nerve diameter"].here_below.all()),
}
for k, x in verdict.items():
    print("%-62s %s" % (k, x))
print("parts holding: %d of %d" % (sum(verdict.values()), len(verdict)))

**Result: partially reproduced (resting on an assumption).** 9 of the 11 parts hold. The
human cervical fascicle numbers (6.67 +/- 4.5) contain all seven published means (5.0 to
9.14); the human cervical nerve area (3.05 mm2) and diameter (1.92 mm) are below every
published mean (3.20 to 8.45 mm2, 2.02 to 3.28 mm), and below 3.0 mm; the human anterior
subdiaphragmatic area and diameter contain the anterior published means; there are fewer
subdiaphragmatic fascicles (4.89) than the posterior study's 16; the pig fascicle numbers
contain the published 46 and 43; the pig diameters (2.20 and 1.64 mm) are below the
published 2.93 and 2.09 mm. Two parts fail: the human cervical endoneurial area,
0.922 +/- 0.337 mm2, does not contain Seki et al.'s 1.32 and 1.84 mm2 (it contains Verlinden
et al.'s 0.75 and 1.09); and the sample range of human cervical nerve area (0.83 to 5.09 mm2)
does not reach Verlinden et al.'s right-side spread (6.0 to 10.9 mm2). The judgement rules
are the claims file's reading of Figure 7.

In [ ]:
rparts = {}
for name, key, lv in [("rat cervical nerve diameter", "diam", "Cervical"), ("rat cervical nerve area", "area", "Cervical"),
                      ("rat subdiaphragmatic nerve diameter", "diam", "Abdominal"), ("rat subdiaphragmatic nerve area", "area", "Abdominal")]:
    t = lit_table(key, "Rat", lv)
    rparts[name] = t
    print("=====", name)
    print(t.round(4).to_string())
rv = {"cervical diameters within mean +/- SD": bool(rparts["rat cervical nerve diameter"].within_mean_sd.all()),
      "cervical areas within mean +/- SD": bool(rparts["rat cervical nerve area"].within_mean_sd.all()),
      "subdiaphragmatic diameter above every published mean": bool(rparts["rat subdiaphragmatic nerve diameter"].here_below.eq(False).all()),
      "subdiaphragmatic area above every published mean": bool(rparts["rat subdiaphragmatic nerve area"].here_below.eq(False).all())}
for k, x in rv.items():
    print("%-55s %s" % (k, x))
for name in ["rat cervical nerve diameter", "rat cervical nerve area"]:
    t = rparts[name]
    print("%s: published means %d of %d within mean +/- SD; within the sample range: %d of %d" % (
        name, t.within_mean_sd.sum(), len(t), ((t.lit_mean >= t.here_min) & (t.lit_mean <= t.here_max)).sum(), len(t)))

**Result: partially reproduced (resting on an assumption).** The rat subdiaphragmatic
nerves are larger than every published value (diameter 0.135 mm against 0.086 to 0.109 mm,
area 0.0145 against 0.0058 to 0.0093 mm2). The cervical part fails under the claims file's
rule: only one of four published diameter means (0.281 mm, Licursi de Alcantara et al.,
left) and one of four area means lie within the rat mean +/- SD (0.258 +/- 0.025 mm,
0.0526 +/- 0.0102 mm2); the others are 0.226, 0.284 and 0.299 mm. Three of the four lie within
the range of the nine samples.

## Perineurium thickness against fascicle diameter

Non-peanut fascicles only (Figure 9 legend). Human from SPARC:65's traces, pig from
SPARC:89's `PigIFMorphology.mat`, rat from this dataset's traces. Fits by linear least
squares (Methods); `DataPlotting_v4.m` fits `poly1` (thickness = p1 d + p2) to every rat
fascicle with a thickness, at both levels pooled, and the pig file stores its own fit's
goodness of fit (282 residual degrees of freedom, so 284 points and two coefficients).

In [ ]:
pr = []
for lab, r in hum.items():
    lv = hsub.set_index("label").loc[lab, "level"]
    for d, t in zip(r["d_fasc_nonpeanut"], r["thk_peri"]):
        pr.append(dict(species="human", sample=lab, level=lv, d=d, thk=t, largest=False))
for d, t, lv in zip(if_d, if_t, pigif.level):
    for dd_, tt_ in zip(d, t):
        pr.append(dict(species="pig", sample=None, level=lv, d=dd_, thk=tt_, largest=False))
for lab, r in rat.items():
    lv = smp.set_index("label").loc[lab, "level"]
    k = v.largest_fascicle(r)["index"]
    for i, (d, t) in enumerate(zip(r["d_fasc_nonpeanut"], r["thk_peri"])):
        pr.append(dict(species="rat", sample=lab, level=lv, d=d, thk=t, largest=(i == k)))
PR = pd.DataFrame(pr)
PR["ratio"] = PR["thk"] / PR["d"]
print(PR.groupby(["species", "level"]).agg(n=("d", "size"), d_min=("d", "min"), d_max=("d", "max"), thk_min=("thk", "min"),
      thk_max=("thk", "max"), ratio_median=("ratio", "median")).round(3))
fits = {sp: v.fit_poly1(PR[PR.species == sp].d, PR[PR.species == sp].thk) for sp in ["human", "pig", "rat"]}
for sp, f in fits.items():
    print("%s: n = %d, thk = %.5f d + %.4f, R2 = %.4f" % (sp, f["n"], f["p1"], f["p2"], f["r2"]))

## rat-perineurium-fit

> The fit of perineurium thickness against fascicle diameter for 22 rat fascicles has
> R2 = 0.77.

In [ ]:
import numpy.polynomial.polynomial as P
def r2_quad(x, y):
    c = np.polyfit(x, y, 2)
    return 1 - ((y - np.polyval(c, x)) ** 2).sum() / ((y - y.mean()) ** 2).sum(), c
rows_rf = []
for setname, sub in [("(a) all fascicles", PR[PR.species == "rat"]), ("(b) 18 largest", PR[(PR.species == "rat") & PR.largest])]:
    f1 = v.fit_poly1(sub.d, sub.thk)
    f0 = v.fit_poly1(sub.d, sub.thk, intercept=False)
    r2q, cq = r2_quad(sub.d.values, sub.thk.values)
    rows_rf.append(dict(set=setname, n=len(sub), r2_poly1=f1["r2"], p1=f1["p1"], p2=f1["p2"], r2_zero=f0["r2"], slope_zero=f0["p1"], r2_quadratic=r2q))
RF = pd.DataFrame(rows_rf)
print(RF.round(4).to_string())
print("expected n = 22 and R2 = 0.77 +/- 0.077")

**Result: reproduced.** Every rat fascicle with a thickness, 22 (reading a), fitted by
thickness = 0.01292 d + 1.367 um (the code's `poly1`), gives R2 = 0.765, within 0.77 +/- 0.077.
With the 18 largest fascicles R2 = 0.664 (n = 18); through zero R2 = 0.510; a quadratic gives
0.767.

## human-perineurium-fit

> The fit of perineurium thickness against fascicle diameter for 73 human non-peanut
> fascicles has R2 = 0.75.

In [ ]:
hh = PR[PR.species == "human"]
for name, f in [("poly1", fits["human"]), ("zero intercept", v.fit_poly1(hh.d, hh.thk, intercept=False))]:
    print("human %s: n = %d, p1 = %.5f, p2 = %.4f, R2 = %.4f" % (name, f["n"], f["p1"], f["p2"], f["r2"]))
print("human quadratic R2 = %.4f" % r2_quad(hh.d.values, hh.thk.values)[0])
print("expected n = 73 and R2 = 0.75 +/- 0.075")

**Result: reproduced.** 73 human non-peanut fascicles; thickness = 0.03702 d + 10.505 um
with R2 = 0.749, within 0.75 +/- 0.075. Through zero R2 = 0.168; a quadratic gives 0.787.

## pig-perineurium-fit

> The fit of perineurium thickness against fascicle diameter for 284 pig non-peanut
> fascicles (IF) has R2 = 0.30.

In [ ]:
pp = PR[PR.species == "pig"]
for name, f in [("poly1", fits["pig"]), ("zero intercept", v.fit_poly1(pp.d, pp.thk, intercept=False))]:
    print("pig %s: n = %d, p1 = %.5f, p2 = %.4f, R2 = %.4f" % (name, f["n"], f["p1"], f["p2"], f["r2"]))
print("pig quadratic R2 = %.4f" % r2_quad(pp.d.values, pp.thk.values)[0])
print("PigIFMorphology.mat's own fit: R2 = %.4f, dfe = %d" % (pif["pigIF_thkperi_gof"]["rsquare"], pif["pigIF_thkperi_gof"]["dfe"]))
print("expected n = 284 and R2 = 0.30 +/- 0.03")

**Result: reproduced.** 284 pig non-peanut IF fascicles; thickness = 0.02547 d + 3.440 um
with R2 = 0.298, within 0.30 +/- 0.03, the value the paper's file stores (0.2984, 282 residual
degrees of freedom). Through zero R2 = -0.007; a quadratic gives 0.301.

## perineurium-species-order

> For a given fascicle diameter, human fascicles have the thickest perineurium and rat
> fascicles the thinnest.

In [ ]:
lo = max(PR[PR.species == sp].d.min() for sp in fits)
hi = min(PR[PR.species == sp].d.max() for sp in fits)
grid = np.linspace(lo, hi, 200)
pred = {sp: f["p1"] * grid + f["p2"] for sp, f in fits.items()}
order_a = bool(np.all(pred["human"] > pred["pig"]) and np.all(pred["pig"] > pred["rat"]))
at250 = {sp: f["p1"] * 250 + f["p2"] for sp, f in fits.items()}
print("common diameter range %.1f to %.1f um" % (lo, hi))
print("(a) human > pig > rat over the whole range:", order_a, "| smallest gaps: human-pig %.2f, pig-rat %.2f um" %
      ((pred["human"] - pred["pig"]).min(), (pred["pig"] - pred["rat"]).min()))
print("(b) fitted thickness at 250 um:", {k: round(x, 2) for k, x in at250.items()}, "| order holds:", at250["human"] > at250["pig"] > at250["rat"])
near = PR[(PR.d > 200) & (PR.d < 300)].groupby("species").thk.agg(["size", "median"])
print("measured thickness of fascicles 200 to 300 um:")
print(near.round(2))

**Result: reproduced.** Over the diameters common to all three species (32.0 to 284.1 um)
the human fit lies above the pig fit (by at least 7.43 um) and the pig fit above the rat fit
(by at least 2.47 um); at 250 um the fits give 19.76, 9.81 and 4.60 um. The measured
thicknesses of fascicles of 200 to 300 um have medians 19.64 (human, 11), 9.93 (pig, 23) and
4.54 um (rat, 9).

## perineurium-ratio-decreases

> The ratio of perineurium thickness to fascicle diameter decreases as fascicle diameter
> increases, particularly in humans and pigs.

In [ ]:
for sp, f in fits.items():
    sub = PR[PR.species == sp]
    g = np.linspace(sub.d.min(), sub.d.max(), 100)
    rfit = (f["p1"] * g + f["p2"]) / g
    rho, p = stats.spearmanr(sub.d, sub.ratio)
    print("%s: (a) fitted ratio %.4f at %.0f um to %.4f at %.0f um, decreasing: %s | (b) Spearman rho = %.3f, p = %.2g, n = %d" %
          (sp, rfit[0], g[0], rfit[-1], g[-1], bool(np.all(np.diff(rfit) < 0)), rho, p, len(sub)))

**Result: reproduced.** The fitted ratio of thickness to diameter decreases over each
species' range (human 1.58 to 0.047, pig 0.265 to 0.037, rat 0.056 to 0.018), and the ratio's
Spearman correlation with diameter is negative (human rho = -0.919, pig -0.601, rat -0.779).

## perineurium-thicker-larger-fascicles

> In the human and pig vagus nerves, larger fascicles have thicker perineurium.

In [ ]:
for sp, f in fits.items():
    sub = PR[PR.species == sp]
    rho, p = stats.spearmanr(sub.d, sub.thk)
    print("%s: (a) slope %.5f (95%% CI %.5f to %.5f), increasing: %s | (b) Spearman rho = %.3f, p = %.2g" %
          (sp, f["p1"], f["ci"][0, 0], f["ci"][0, 1], f["p1"] > 0, rho, p))

**Result: reproduced.** The fitted slopes are positive (human 0.0370, 95% CI 0.0320 to
0.0421; pig 0.0255, 0.0209 to 0.0300) and the Spearman correlations of thickness with
diameter are positive (0.918 and 0.545). Rats too (slope 0.0129, rho 0.864).

## perineurium-no-level-difference

> The relationship between perineurium thickness and fascicle diameter does not differ
> between cervical and subdiaphragmatic nerves in any of the three species.

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
nld = {}
for sp in ["human", "pig", "rat"]:
    sub = PR[PR.species == sp]
    fc = v.fit_poly1(sub[sub.level == "cervical"].d, sub[sub.level == "cervical"].thk)
    fa = v.fit_poly1(sub[sub.level == "subdiaphragmatic"].d, sub[sub.level == "subdiaphragmatic"].thk)
    overlap = [bool(fc["ci"][i, 0] <= fa["ci"][i, 1] and fa["ci"][i, 0] <= fc["ci"][i, 1]) for i in range(2)]
    m0 = smf.ols("thk ~ d", sub).fit()
    m1 = smf.ols("thk ~ d * C(level)", sub).fit()
    ft = anova_lm(m0, m1)
    nld[sp] = (all(overlap), float(ft["Pr(>F)"].iloc[1]))
    print("%s: cervical n = %d p1 %.4f [%.4f, %.4f] p2 %.3f [%.3f, %.3f] | subdiaphragmatic n = %d p1 %.4f [%.4f, %.4f] p2 %.3f [%.3f, %.3f]" %
          (sp, fc["n"], fc["p1"], *fc["ci"][0], fc["p2"], *fc["ci"][1], fa["n"], fa["p1"], *fa["ci"][0], fa["p2"], *fa["ci"][1]))
    print("    (a) bounds overlap, slope %s, intercept %s | (b) F test of level terms F = %.3f, p = %.3g" %
          (overlap[0], overlap[1], ft["F"].iloc[1], ft["Pr(>F)"].iloc[1]))

**Result: not reproduced (resting on an assumption).** Under reading (a), the 95% bounds of
the cervical and subdiaphragmatic fits do not overlap for the human slope (0.0318 [0.0248,
0.0388] against 0.0558 [0.0494, 0.0622]) nor for the pig intercept (5.26 [4.14, 6.39] against
3.16 [2.67, 3.65] um); they overlap for both rat coefficients. The F test of the level terms
(reading b) agrees: human p = 2.4e-07, pig p = 3.1e-09, rat p = 0.87. The tests are the claims
file's, as the paper names none.

## sunderland-reanalysis

> Refitting all 777 fascicles of Sunderland and Bradley (1952) gives a zero-intercept slope
> of 2.75% of fascicle diameter for perineurium thickness, with R2 = 0.51.

The table is `files/code/Sunderland1984.mat` of SPARC:65 (fascicle diameter and
perineurium thickness, um), with its plotting script `Sunderland1984_Plotting.m`.

In [ ]:
sund = v.mat("65", "files/code/Sunderland1984.mat")
print(v.read("65", "files/code/Sunderland1984_Plotting.m").decode("latin-1")[:1600])
sx, sy = sund["dfasc_um"].astype(float), sund["thk_peri_um"].astype(float)
f0 = v.fit_poly1(sx, sy, intercept=False)
print("n = %d, d %.0f to %.0f um, thickness %.2f to %.2f um" % (len(sx), sx.min(), sx.max(), sy.min(), sy.max()))
print("zero-intercept slope %.4f%% | R2 (about the mean, as MATLAB fit) %.4f | R2 uncentred %.4f" % (100 * f0["p1"], f0["r2"], f0["r2_uncentered"]))
print("ratio of thickness to diameter: mean %.2f%% SD %.2f%%" % (100 * np.mean(sy / sx), 100 * np.std(sy / sx, ddof=1)))

**Result: reproduced.** The 777 fascicles of the table give a zero-intercept slope of
2.750% with R2 = 0.510 (about the mean, as MATLAB's fit computes it), the paper's 2.75% and
0.51; the uncentred R2 would be 0.853.

## rat-perineurium-near-3-percent

> Rat perineurium thickness is close to the published 3% of fascicle diameter, slightly
> thicker for smaller fascicles and thinner for larger ones.

In [ ]:
fr_ = fits["rat"]
sub = PR[PR.species == "rat"]
g = np.linspace(sub.d.min(), sub.d.max(), 200)
for slope in [0.03, 0.0275]:
    q = (fr_["p1"] * g + fr_["p2"]) / (slope * g)
    print("reference %.2f%%: fit over reference %.3f at %.1f um to %.3f at %.1f um | within 0.5 to 2: %s | above at smallest: %s, below at largest: %s" %
          (100 * slope, q[0], g[0], q[-1], g[-1], bool(np.all((q > 0.5) & (q < 2))), q[0] > 1, q[-1] < 1))
print("crossing diameter, fit = 3%%: %.1f um" % (fr_["p2"] / (0.03 - fr_["p1"])))

**Result: reproduced (resting on an assumption).** Against the 3% line, the rat fit over the line runs from 1.856 at
32.0 um to 0.591 at 284.1 um, within a factor of two, above the line for small fascicles and
below for large ones (crossing at 80.1 um). Against the 2.75% line the ratio reaches 2.024 at
the smallest diameter, just outside a factor of two, so the claim holds under the 3% reading
and not quite under the 2.75% one.

## human-pig-perineurium-thicker

> Human and pig vagal perineurium is substantially thicker than the 3% of fascicle diameter
> found for human somatic nerves, especially for smaller fascicles.

In [ ]:
for sp in ["human", "pig", "rat"]:
    sub = PR[PR.species == sp]
    med = sub.d.median()
    exc_s = (sub[sub.d < med].ratio - 0.03).median()
    exc_l = (sub[sub.d >= med].ratio - 0.03).median()
    print("%s: median ratio %.4f | median excess over 0.03, below median diameter %.4f, above %.4f | holds: %s" %
          (sp, sub.ratio.median(), exc_s, exc_l, sub.ratio.median() > 0.03 and exc_s > exc_l))

**Result: reproduced (resting on an assumption).** The median ratio of thickness to diameter is 0.0948 in humans and
0.0527 in pigs, above 0.03, and the median excess over 0.03 is larger below the species'
median diameter than above it (human 0.1061 against 0.0379, pig 0.0329 against 0.0160). Rats
are below the line (median 0.0214).

## pig-if-mt-consistent

> Pig fascicle diameters from anti-fibronectin IF (14 to 303 um, mean 123, median 123,
> interquartile range 91 to 155 um) are consistent with those from MT (5 to 399 um, mean
> 135, median 130, interquartile range 96 to 168 um).

In [ ]:
def summ(x):
    x = np.asarray(x)
    return dict(n=len(x), min=x.min(), max=x.max(), mean=x.mean(), median=np.median(x),
                q25=np.percentile(x, 25), q75=np.percentile(x, 75),
                q25_matlab=np.percentile(x, 25, method="hazen"), q75_matlab=np.percentile(x, 75, method="hazen"))
ifx = np.concatenate(if_d)
SU = pd.DataFrame([dict(set="IF non-peanut", **summ(ifx)), dict(set="IF all inner traces", **summ(np.concatenate(if_dall))),
                   dict(set="MT all inner traces", **summ(allpig))])
print(SU.round(1).to_string())
def widen(lo_, hi_):
    return lo_ - 0.1 * lo_, hi_ + 0.1 * hi_
checks = {"IF mean": abs(SU.loc[0, "mean"] - 123) <= 12.3, "IF median": abs(SU.loc[0, "median"] - 123) <= 12.3,
          "MT mean": abs(SU.loc[2, "mean"] - 135) <= 13.5, "MT median": abs(SU.loc[2, "median"] - 130) <= 13}
for name, row, (a_, b_), (c_, d_) in [("IF", 0, (14, 303), (91, 155)), ("MT", 2, (5, 399), (96, 168))]:
    w1, w2 = widen(a_, b_), widen(c_, d_)
    checks[name + " min"] = w1[0] <= SU.loc[row, "min"] <= w1[1]
    checks[name + " max"] = w1[0] <= SU.loc[row, "max"] <= w1[1]
    checks[name + " IQR"] = w2[0] <= SU.loc[row, "q25"] and SU.loc[row, "q75"] <= w2[1]
print(checks)

**Result: reproduced.** IF non-peanut fascicles (284): 14.4 to 302.8 um, mean 123.3,
median 123.1, interquartile range 88.1 to 152.5 um; MT fascicles (766): 23.7 to 399.1 um,
mean 134.6, median 129.5, interquartile range 95.7 to 167.7 um. Means and medians are within
10%, and the ranges and interquartile ranges lie within the reported ones widened by 10% of
each bound. The MT minimum here is 23.7 um where the paper gives 5 um: the output file holds
no fascicle below 23.7 um.

## peanut-fascicle-counts

> Human IHC micrographs have 60 cervical inner perineurium traces (27 bundles in 10 peanut
> fascicles) and 44 subdiaphragmatic (4 in 2); pig IF micrographs have 210 cervical (57 in
> 25) and 179 subdiaphragmatic (48 in 19).

In [ ]:
hc = pd.DataFrame([dict(level=hsub.set_index("label").loc[l, "level"], inner=r["n_fasc"], bundles=r["n_peanut_inner"],
                        peanuts=r["n_peanut_outer"], nonpeanut=r["n_nonpeanut"]) for l, r in hum.items()]).groupby("level").sum()
pc = pit.groupby("level")[["n_inner", "n_peanut_inner", "n_peanut_outer", "n_nonpeanut"]].sum()
pc.columns = ["inner", "bundles", "peanuts", "nonpeanut"]
pcc = pit.groupby("level")[["code_inner", "code_bundles", "code_peanut_outer", "code_nonpeanut", "code_inner_holes", "code_outer_empty"]].sum()
print("human (SPARC:65 traces):"); print(hc)
print("pig IF (SPARC:89 traces, objects only, as for the other species):"); print(pc)
print("pig IF (SPARC:89 traces, counted as SegmentationAnalysis_v2.m counts them, hole boundaries included):"); print(pcc)
pf_ = pigif.groupby("level")[["n_inner", "n_nonpeanut"]].sum()
print("pig IF (PigIFMorphology.mat):"); print(pf_)
print("expected: human 60 (27 in 10), 44 (4 in 2); pig 210 (57 in 25), 179 (48 in 19); non-peanut sums 73 and 284")
print("non-peanut sums: human %d, pig traces %d (as the IF code counts, %d), pig file %d" % (hc.nonpeanut.sum(), pc.nonpeanut.sum(),
      pcc.code_nonpeanut.sum(), pf_.n_nonpeanut.sum()))

**Result: partially reproduced.** Human traces: 60 cervical inner traces with 27 bundles in
10 peanut fascicles and 44 subdiaphragmatic with 4 in 2, as the legend says. Pig IF traces,
counted as SPARC:89's code counts them: 210 cervical with 57 bundles in 24 peanut fascicles
(the legend says 25) and 179 subdiaphragmatic with 48 in 19, as the legend says. 11 of the 12
counts match. The non-peanut sums are 73 (human) and 284 (pig), the fits' counts.

## no-age-trend

> The morphology metrics show no trend with age in any species.

Per species and level, each per-sample metric against the subject's age: nerve diameter,
number of fascicles (human and pig), mean fascicle diameter, endoneurium proportion and
perineurium proportion (human and rat). Human ages "90+ years" are entered as 99, as the
paper's code does (DataExtraction_v3.m). Pig ages take four values (10.5 to 15 weeks).

In [ ]:
ag = []
for sp in ["human", "pig", "rat"]:
    for lv in ["cervical", "subdiaphragmatic"]:
        sub = S[(S.species == sp) & (S.level == lv)]
        for col in ["d_nerve", "n_fasc", "mean_fasc", "endo_pct", "peri_pct"]:
            if (col == "n_fasc" and sp == "rat") or (col == "peri_pct" and sp == "pig"):
                continue
            x, y = sub.age.astype(float).values, sub[col].astype(float).values
            r_, p_ = stats.pearsonr(x, y)
            rs, ps = stats.spearmanr(x, y)
            ag.append(dict(species=sp, level=lv, metric=col, n=len(x), pearson_r=r_, pearson_p=p_, spearman_rho=rs, spearman_p=ps))
AG = pd.DataFrame(ag)
print(AG.round(4).to_string())
print("tests: %d; Pearson p < 0.05: %d; Spearman p < 0.05: %d" % (len(AG), (AG.pearson_p < 0.05).sum(), (AG.spearman_p < 0.05).sum()))
print("expected by chance at 0.05: %.1f" % (0.05 * len(AG)))
print(AG[(AG.pearson_p < 0.05) | (AG.spearman_p < 0.05)].round(4).to_string())

**Result: not reproduced (resting on an assumption).** Of 26 correlations with age, three
have Pearson p < 0.05 (reading a), all in pigs: cervical endoneurium proportion r = -0.72
(p = 0.029), subdiaphragmatic nerve diameter r = 0.75 (p = 0.019) and subdiaphragmatic mean
fascicle diameter r = 0.69 (p = 0.039); by Spearman (reading b) one, human subdiaphragmatic
mean fascicle diameter (rho = -0.69, p = 0.041). 1.3 would be expected by chance at 0.05;
the tests are uncorrected, as the claims file names them, and the pig ages take four values
(10.5 to 15 weeks). Supplement 6, in the supplementary PDF of Europe PMC, is a figure of the
metrics against age with no test, so the paper's judgement is visual.

## rat-carotid-sheath-bundles

> The rat cervical carotid sheath holds up to six fascicles, of which the largest, taken as
> the vagus, was segmented.

In [ ]:
print("rat cervical traces: outer perineurium traces per sample", {l: rat[l]["n_outer"] for l in smp[smp.level == "cervical"]["label"]})

**Result: blocked (data missing).** Each cervical PeriMask holds only the vagus (one outer
and one inner trace); the other bundles of the carotid sheath are not traced. The
derivative folder has the micrographs (JPEG 2000), but counting their bundles needs a
segmentation the dataset does not hold.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| rat-sample-set (link) | reproduced |
| nerve-diameter-anova | reproduced, resting on an assumption (unbounded variance components) |
| human-pig-nerve-diameter-similar | reproduced |
| rat-nerve-diameter | reproduced |
| rat-ten-times-smaller | reproduced |
| nerve-cervical-larger | reproduced, resting on an assumption (reading of "clearest") |
| fascicle-number-human-pig | reproduced |
| rat-subdiaphragmatic-multifascicular | reproduced |
| rat-multifascicular-proportions | reproduced |
| fascicle-diameter-anova | reproduced |
| rat-fascicle-diameter | reproduced |
| cervical-fascicle-diameter-species | reproduced |
| subdiaphragmatic-fascicle-diameter-comparable | reproduced |
| fascicles-larger-cervical | reproduced, resting on an assumption (degrees of freedom) |
| rat-diameter-like-fascicle | reproduced |
| human-pig-tissue-proportions | reproduced |
| rat-level-ratios | reproduced (under the connective tissue reading only) |
| human-pig-level-ratios | reproduced |
| human-variability | reproduced, resting on an assumption (the measure) |
| literature-comparison-human-pig | partially reproduced, resting on an assumption (9 of 11 parts) |
| rat-literature-comparison | partially reproduced, resting on an assumption (cervical part fails) |
| perineurium-species-order | reproduced |
| perineurium-ratio-decreases | reproduced |
| human-perineurium-fit | reproduced |
| pig-perineurium-fit | reproduced |
| rat-perineurium-fit | reproduced |
| perineurium-thicker-larger-fascicles | reproduced |
| perineurium-no-level-difference | not reproduced, resting on an assumption (the test) |
| sunderland-reanalysis | reproduced |
| rat-perineurium-near-3-percent | reproduced, resting on an assumption (factor of two) |
| human-pig-perineurium-thicker | reproduced, resting on an assumption (the measure) |
| pig-if-mt-consistent | reproduced |
| peanut-fascicle-counts | partially reproduced (pig cervical peanut fascicles 24, not 25) |
| no-age-trend | not reproduced, resting on an assumption (the test) |
| rat-carotid-sheath-bundles | blocked, data missing (only the vagus is traced) |

**How the analysis differs from the paper's.** The morphometry repeats the paper's MATLAB
code in Python and gives its rat and human values to rounding. The rat "largest fascicle"
is the largest by endoneurial area, as the Methods say; the code's first inner trace, which
differs in R22-1 and R23-1, is a variant. The mixed models are fitted by REML with
unbounded variance components, JMP's default, and tested with Satterthwaite degrees of
freedom in place of JMP's Kenward-Roger; the F values are within 4% of the paper's for nerve
diameter and fascicle number. The pig MT and IF values come from the per-fascicle output
files of SPARC:64 and SPARC:89, not from their traces (the IF traces are used for counts).
Human and pig data come from the sister datasets SPARC:65, SPARC:64 and SPARC:89, and the
literature values from Supplement 4. Where the paper names no test (the level comparison of
the perineurium fits, the age trends, the literature comparisons), the claims file's tests
are used.

**What was found about the dataset.** The traces of all 18 rat samples are there, and
measuring them with the paper's method gives the numbers of the code folder's output file
exactly, and the paper's. The derivative folder's `RatVNMorphology_OutputMetrics.mat` is an
older output (2020-03-03): it holds all three species (54 samples), its rat values are not
those of the traces (cervical nerve diameters up to 8.5 um larger) and its manifest entry
describes it as rat metrics. The code folder's output file keeps only 7 of the 22 rat
perineurium thicknesses. The readme gives ages 65 to 268 days where subjects.xlsx gives 75
to 268. samples.xlsx does not say that the subdiaphragmatic samples are anterior. In
SPARC:65 the NerveMask of C54-3 is inverted, so the published code would measure the image
frame (3.57 mm) in place of the nerve (1.62 mm).

In [ ]:
print("total time %.0f s, max rss %.2f GB" % (time.time() - T0, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))
print("files read, SHA-256:")
for (ds, p), h in sorted(v.READ.items()):
    where = f"SPARC:{ds} v{v.DATASETS[ds][0]}" if ds in v.DATASETS else f"Europe PMC {ds} supplementary files"
    print(f"  {where} {p} {h}")